# p02: LDA Topic Modeling on Airbnb Reviews by Language

This notebook performs Latent Dirichlet Allocation (LDA) topic modeling on Airbnb review comments, with:
- Separate models trained for each major language
- Sampling of comments per language (configurable)
- Automatic topic number optimization based on coherence metrics
- Extraction of phi (topic-word) and theta (document-topic) matrices
- Preservation of comment IDs for downstream statistical analysis

## Configuration

In [34]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Sample size per language (set to None or 'total' to use all data)
SAMPLE_SIZE_PER_LANGUAGE = 1000

# Topic optimization range
MIN_TOPICS = 3
MAX_TOPICS = 10  # Will test topic numbers from MIN_TOPICS to MAX_TOPICS

# LDA hyperparameters
LDA_PASSES = 10
LDA_ITERATIONS = 100
LDA_WORKERS = 4

# Random seed for reproducibility
RANDOM_SEED = 42

# Energy and Carbon tracking
POWER_WATTS = 100  # Estimated CPU power consumption (Watts)
FR_GRID_KGCO2_PER_KWH = 0.06  # French grid carbon intensity (kg CO2e per kWh)

# Track pipeline start time
PIPELINE_START_TIME = time.time()

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Sample size per language: {SAMPLE_SIZE_PER_LANGUAGE if SAMPLE_SIZE_PER_LANGUAGE else 'ALL'}")
print(f"Topic optimization range: {MIN_TOPICS} to {MAX_TOPICS}")
print(f"LDA passes: {LDA_PASSES}")
print(f"LDA iterations: {LDA_ITERATIONS}")
print(f"Estimated CPU power: {POWER_WATTS}W")
print(f"Grid carbon intensity: {FR_GRID_KGCO2_PER_KWH} kg CO2e/kWh")
print("="*70)

CONFIGURATION
Sample size per language: 1000
Topic optimization range: 3 to 10
LDA passes: 10
LDA iterations: 100
Estimated CPU power: 100W
Grid carbon intensity: 0.06 kg CO2e/kWh


## 1. Import Required Libraries

In [8]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime
from pathlib import Path

# Text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import re
import string

# LDA and topic modeling
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import gensim
from gensim import corpora, models
from gensim.models import LdaModel, CoherenceModel

# Progress bars
from tqdm import tqdm
tqdm.pandas()

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
print("Downloading NLTK data...")
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Load and Explore Review Data

In [25]:
# Load reviews data
print("Loading reviews data...")
data_path = os.path.join('data', 'reviews_select.csv')
df = pd.read_csv(data_path)

print(f"✓ Loaded {len(df):,} reviews")
print(f"\nDataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nFirst rows:")
print(df.head())

# Explore language distribution
print(f"\n\nLanguage distribution:")
lang_distribution = df['langue'].value_counts()
print(lang_distribution)

# Check for missing values in key columns
print(f"\n\nMissing values:")
print(df[['id', 'comments', 'langue']].isnull().sum())

# Get statistics on comment length
print(f"\n\nComment statistics:")
df['comment_length'] = df['comments'].str.len()
print(df['comment_length'].describe())

Loading reviews data...
✓ Loaded 530,818 reviews

Dataset shape: (530818, 10)

Column names:
['Unnamed: 0', 'listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'langue', 'langue_short', 'nwords']

First rows:
   Unnamed: 0  listing_id            id        date  reviewer_id  \
0           1      5396.0  9.164898e+17  2023-06-18  497745629.0   
1           2      5396.0  9.216001e+17  2023-06-25   70206366.0   
2           3      5396.0  9.266632e+17  2023-07-02   41320355.0   
3           4      5396.0  9.288200e+17  2023-07-05  259277676.0   
4           5      5396.0  9.295177e+17  2023-07-06   93503104.0   

  reviewer_name                                           comments langue  \
0        Grazia  Alloggio confortevole e pratico, dotato di tut...     it   
1      Benjamin  Très bon emplacement pour cet appartement typi...     fr   
2         Julia  What a wonderful gem.  Great location, it was ...     en   
3         GLori  We had a lovely 3 night stay.  Everyt

## 3. Define Text Preprocessing Functions

In [10]:
# Define preprocessing functions for each language
LANGUAGE_MAP = {
    'English': 'english',
    'French': 'french',
    'Italian': 'italian',
    'Spanish': 'spanish',
    'German': 'german',
    'Dutch': 'dutch',
    'Portuguese': 'portuguese'
}

def get_stopwords_by_language(language):
    """Get stopwords for a given language."""
    if language in LANGUAGE_MAP:
        lang_key = LANGUAGE_MAP[language]
        try:
            return set(stopwords.words(lang_key))
        except:
            return set(stopwords.words('english'))
    return set(stopwords.words('english'))

def get_stemmer_by_language(language):
    """Get stemmer for a given language."""
    if language in LANGUAGE_MAP:
        lang_key = LANGUAGE_MAP[language]
        try:
            return SnowballStemmer(lang_key)
        except:
            return SnowballStemmer('english')
    return SnowballStemmer('english')

def preprocess_text(text, language='English', min_token_length=3):
    """
    Preprocess text for a specific language.
    
    Parameters:
    - text: input text string
    - language: language name (e.g., 'English', 'French')
    - min_token_length: minimum token length to keep
    
    Returns:
    - list of preprocessed tokens
    """
    if pd.isna(text) or text == '':
        return []
    
    # Convert to lowercase and remove special characters
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()
    
    # Get stopwords and stemmer for the language
    stop_words = get_stopwords_by_language(language)
    stemmer = get_stemmer_by_language(language)
    
    # Filter and stem
    tokens = [
        stemmer.stem(token) 
        for token in tokens 
        if token not in stop_words and len(token) >= min_token_length and token.isalpha()
    ]
    
    return tokens

print("✓ Preprocessing functions defined")

✓ Preprocessing functions defined


## 4. Sample Data by Language

In [27]:
# Identify main languages (select top 8 most frequent)
MIN_SAMPLES_PER_LANGUAGE = 500

language_counts = df['langue'].value_counts()
main_languages = language_counts.head(8).index.tolist()

print(f"Top 8 languages by frequency:")
print("-" * 60)

# Sample and prepare data by language
language_data = {}
sampling_report = []

for lang in main_languages:
    lang_df = df[df['langue'] == lang].copy()
    lang_df = lang_df[lang_df['comments'].notna() & (lang_df['comments'].str.len() > 0)].reset_index(drop=True)
    
    original_count = len(lang_df)
    
    # Apply sampling
    if SAMPLE_SIZE_PER_LANGUAGE and isinstance(SAMPLE_SIZE_PER_LANGUAGE, int):
        if len(lang_df) > SAMPLE_SIZE_PER_LANGUAGE:
            lang_df = lang_df.sample(n=SAMPLE_SIZE_PER_LANGUAGE, random_state=RANDOM_SEED)
    
    sampled_count = len(lang_df)
    sample_pct = (sampled_count / original_count * 100) if original_count > 0 else 0
    
    print(f"\n  {lang}:")
    print(f"    - Available: {original_count:,}")
    print(f"    - Used:      {sampled_count:,} ({sample_pct:.1f}%)")
    
    # Store sampled data
    language_data[lang] = lang_df
    sampling_report.append({
        'langue': lang,
        'available': original_count,
        'sampled': sampled_count,
        'percentage': sample_pct
    })

print("\n" + "="*60)
print(f"✓ Sampling completed: {len(language_data)} languages processed")
print("="*60)

Top 8 languages by frequency:
------------------------------------------------------------

  en:
    - Available: 269,778
    - Used:      1,000 (0.4%)

  fr:
    - Available: 129,519
    - Used:      1,000 (0.8%)

  es:
    - Available: 29,932
    - Used:      1,000 (3.3%)

  de:
    - Available: 22,314
    - Used:      1,000 (4.5%)

  it:
    - Available: 14,522
    - Used:      1,000 (6.9%)

  pt:
    - Available: 10,204
    - Used:      1,000 (9.8%)

  ko:
    - Available: 8,554
    - Used:      1,000 (11.7%)

  nl:
    - Available: 7,538
    - Used:      1,000 (13.3%)

✓ Sampling completed: 8 languages processed


## 5. Preprocess Texts and Create Corpora

In [36]:
print("="*60)
print("PREPROCESSING TEXTS BY LANGUAGE")
print("="*60)

# Dictionary to store prepared data by language
dictionaries = {}
corpora_data = {}
doc_term_matrices = {}

for lang in main_languages:
    print(f"\n{lang}:")
    
    lang_df = language_data[lang]
    print(f"  - Input documents: {len(lang_df):,}")
    
    # Preprocess texts
    print(f"  - Preprocessing texts...")
    lang_df['tokens'] = lang_df['comments'].progress_apply(
        lambda x: preprocess_text(x, language=lang, min_token_length=3)
    )
    
    # Filter out empty documents
    lang_df = lang_df[lang_df['tokens'].apply(len) > 0].reset_index(drop=True)
    print(f"  - Non-empty documents after preprocessing: {len(lang_df):,}")
    
    # Create gensim dictionary
    print(f"  - Creating gensim dictionary...")
    tokens_list = lang_df['tokens'].tolist()
    gensim_dict = corpora.Dictionary(tokens_list)
    
    print(f"    Initial vocabulary size: {len(gensim_dict)}")
    
    # Filter extremes (rare and very common words)
    # no_below=5: eliminate terms appearing in < 5 documents
    # no_above=0.8: eliminate terms appearing in > 80% of documents
    gensim_dict.filter_extremes(
        no_below=5,
        no_above=0.8,
        keep_n=None
    )
    print(f"    Final vocabulary size after filtering: {len(gensim_dict)}")
    
    # Create corpus (bag of words representation)
    print(f"  - Creating BoW corpus...")
    corpus = [gensim_dict.doc2bow(doc) for doc in tokens_list]
    
    # Create document-term matrix (dense format for later use)
    print(f"  - Creating document-term matrix...")
    doc_term_matrix = gensim.matutils.corpus2dense(corpus, num_terms=len(gensim_dict)).T
    
    # Store
    language_data[lang] = lang_df
    dictionaries[lang] = gensim_dict
    corpora_data[lang] = corpus
    doc_term_matrices[lang] = doc_term_matrix
    
    print(f"    Document-term matrix shape: {doc_term_matrix.shape}")

print("\n" + "="*60)
print("✓ Preprocessing completed for all languages")
print("="*60)

PREPROCESSING TEXTS BY LANGUAGE

en:
  - Input documents: 1,000
  - Preprocessing texts...


100%|██████████| 1000/1000 [00:28<00:00, 34.97it/s]


  - Non-empty documents after preprocessing: 1,000
  - Creating gensim dictionary...
    Initial vocabulary size: 2692
    Final vocabulary size after filtering: 712
  - Creating BoW corpus...
  - Creating document-term matrix...
    Document-term matrix shape: (1000, 712)

fr:
  - Input documents: 1,000
  - Preprocessing texts...


100%|██████████| 1000/1000 [00:28<00:00, 34.84it/s]


  - Non-empty documents after preprocessing: 1,000
  - Creating gensim dictionary...
    Initial vocabulary size: 2710
    Final vocabulary size after filtering: 592
  - Creating BoW corpus...
  - Creating document-term matrix...
    Document-term matrix shape: (1000, 592)

es:
  - Input documents: 1,000
  - Preprocessing texts...


...

  - Non-empty documents after preprocessing: 1,000
  - Creating gensim dictionary...
    Initial vocabulary size: 3955
    Final vocabulary size after filtering: 851
  - Creating BoW corpus...
  - Creating document-term matrix...
    Document-term matrix shape: (1000, 851)

ko:
  - Input documents: 994
  - Preprocessing texts...


100%|██████████| 994/994 [00:28<00:00, 34.74it/s]


  - Non-empty documents after preprocessing: 994
  - Creating gensim dictionary...
    Initial vocabulary size: 9754
    Final vocabulary size after filtering: 664
  - Creating BoW corpus...
  - Creating document-term matrix...
    Document-term matrix shape: (994, 664)

nl:
  - Input documents: 1,000
  - Preprocessing texts...


100%|██████████| 1000/1000 [00:28<00:00, 34.76it/s]


  - Non-empty documents after preprocessing: 1,000
  - Creating gensim dictionary...
    Initial vocabulary size: 3363
    Final vocabulary size after filtering: 744
  - Creating BoW corpus...
  - Creating document-term matrix...
    Document-term matrix shape: (1000, 744)

✓ Preprocessing completed for all languages


## 6. Optimize Number of Topics (Coherence Analysis)

In [37]:
print("="*70)
print("OPTIMIZING NUMBER OF TOPICS")
print("="*70)
print(f"\nTesting topic numbers from {MIN_TOPICS} to {MAX_TOPICS}...")

# Dictionary to store optimization results
optimal_topics = {}
coherence_results = {}

for lang in main_languages:
    print(f"\n{'='*70}")
    print(f"{lang}:")
    print(f"{'='*70}")
    
    corpus = corpora_data[lang]
    dictionary = dictionaries[lang]
    
    coherence_scores = []
    topic_range = range(MIN_TOPICS, MAX_TOPICS + 1)
    
    for num_topics in tqdm(topic_range, desc=f"Testing topics", leave=True):
        # Train LDA model
        lda_model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=RANDOM_SEED,
            passes=LDA_PASSES,
            iterations=LDA_ITERATIONS,
            per_word_topics=True,
            minimum_probability=0.0
        )
        
        # Calculate coherence score (lower/more negative is better for U_Mass)
        coherence_model = CoherenceModel(
            model=lda_model,
            corpus=corpus,
            dictionary=dictionary,
            coherence='u_mass'
        )
        coherence = coherence_model.get_coherence()
        coherence_scores.append(coherence)
        
        print(f"  Topics: {num_topics:2d} → Coherence (U_Mass): {coherence:8.4f}")
    
    # Find optimal number of topics (highest coherence, i.e., closest to 0 or least negative)
    best_idx = np.argmin(np.abs(coherence_scores))  # Find closest to 0
    best_num_topics = list(topic_range)[best_idx]
    best_coherence = coherence_scores[best_idx]
    
    optimal_topics[lang] = best_num_topics
    coherence_results[lang] = {
        'topic_range': list(topic_range),
        'coherence_scores': coherence_scores,
        'optimal_topics': best_num_topics,
        'optimal_coherence': best_coherence
    }
    
    print(f"\n  ✓ OPTIMAL: {best_num_topics} topics (Coherence: {best_coherence:.4f})")

print("\n" + "="*70)
print("TOPIC OPTIMIZATION SUMMARY")
print("="*70)
for lang in main_languages:
    print(f"  {lang:15s}: {optimal_topics[lang]} topics")
print("="*70)

OPTIMIZING NUMBER OF TOPICS

Testing topic numbers from 3 to 10...

en:


Testing topics:  12%|█▎        | 1/8 [00:21<02:27, 21.05s/it]

  Topics:  3 → Coherence (U_Mass):  -1.4950


Testing topics:  25%|██▌       | 2/8 [00:36<01:48, 18.01s/it]

  Topics:  4 → Coherence (U_Mass):  -1.5777


...

  Topics:  8 → Coherence (U_Mass):  -1.4553


Testing topics:  88%|████████▊ | 7/8 [02:01<00:16, 16.96s/it]

  Topics:  9 → Coherence (U_Mass):  -1.5207


Testing topics: 100%|██████████| 8/8 [02:15<00:00, 16.98s/it]

  Topics: 10 → Coherence (U_Mass):  -1.5451

  ✓ OPTIMAL: 6 topics (Coherence: -1.3316)

TOPIC OPTIMIZATION SUMMARY
  en             : 3 topics
  fr             : 5 topics
  es             : 5 topics
  de             : 3 topics
  it             : 3 topics
  pt             : 3 topics
  ko             : 5 topics
  nl             : 6 topics


## 7. Train Final LDA Models with Optimized Topics

In [38]:
print("="*70)
print("TRAINING FINAL LDA MODELS")
print("="*70)

# Dictionary to store trained models
lda_models = {}
final_coherence_scores = {}

for lang in main_languages:
    print(f"\n{lang}:")
    
    num_topics = optimal_topics[lang]
    print(f"  - Training LDA with {num_topics} topics...")
    
    start_time = time.time()
    
    # Train final LDA model with optimized number of topics
    lda_model = LdaModel(
        corpus=corpora_data[lang],
        id2word=dictionaries[lang],
        num_topics=num_topics,
        random_state=RANDOM_SEED,
        passes=LDA_PASSES,
        iterations=LDA_ITERATIONS,
        per_word_topics=True,
        minimum_probability=0.0
    )
    
    training_time = time.time() - start_time
    print(f"    Training time: {training_time:.2f}s")
    
    # Calculate final coherence score
    print(f"  - Calculating final coherence...")
    coherence_model = CoherenceModel(
        model=lda_model,
        corpus=corpora_data[lang],
        dictionary=dictionaries[lang],
        coherence='u_mass'
    )
    coherence_score = coherence_model.get_coherence()
    final_coherence_scores[lang] = coherence_score
    
    print(f"    Coherence (U_Mass): {coherence_score:.4f}")
    
    # Store model
    lda_models[lang] = lda_model
    
    # Display top topics
    print(f"\n  - Top {min(5, num_topics)} topics:")
    topics = lda_model.show_topics(num_topics=min(5, num_topics), num_words=5)
    for topic_id, topic_words in topics:
        print(f"    Topic {topic_id}: {topic_words}")

print("\n" + "="*70)
print("✓ Final LDA models trained successfully")
print("="*70)

TRAINING FINAL LDA MODELS

en:
  - Training LDA with 3 topics...
    Training time: 19.33s
  - Calculating final coherence...
    Coherence (U_Mass): -1.4950

  - Top 3 topics:
    Topic 0: 0.032*"apart" + 0.028*"stay" + 0.027*"locat" + 0.020*"good" + 0.015*"clean"
    Topic 1: 0.025*"apart" + 0.025*"stay" + 0.018*"place" + 0.016*"great" + 0.015*"pari"
    Topic 2: 0.061*"great" + 0.038*"stay" + 0.033*"place" + 0.031*"locat" + 0.026*"apart"

fr:
  - Training LDA with 5 topics...
    Training time: 14.21s
  - Calculating final coherence...
    Coherence (U_Mass): -1.4745

  - Top 5 topics:
    Topic 0: 0.113*"très" + 0.061*"bien" + 0.051*"est" + 0.035*"appart" + 0.027*"séjour"
    Topic 1: 0.044*"est" + 0.040*"nous" + 0.038*"très" + 0.032*"appart" + 0.032*"pour"
    Topic 2: 0.043*"les" + 0.042*"est" + 0.039*"très" + 0.023*"appart" + 0.022*"pour"
    Topic 3: 0.040*"est" + 0.034*"pas" + 0.029*"pour" + 0.023*"très" + 0.022*"des"
    Topic 4: 0.034*"recommand" + 0.033*"logement" + 0.031*"

## 8. Extract Phi and Theta Matrices

In [39]:
def extract_phi_matrix(lda_model, num_topics, dictionary, vocab_size):
    """
    Extract phi matrix (topic-word distribution) from LDA model.
    
    Returns:
    - phi: 2D numpy array of shape (num_topics, vocab_size)
           phi[i,j] = probability of word j given topic i
    """
    phi = np.zeros((num_topics, vocab_size))
    
    for topic_id in range(num_topics):
        terms = lda_model.show_topic(topic_id, topn=vocab_size)
        for term, prob in terms:  # term is a string, not an ID
            term_id = dictionary.token2id.get(term, -1)
            if term_id >= 0:  # Only add if term exists in dictionary
                phi[topic_id, term_id] = prob
    
    return phi

def extract_theta_matrix(lda_model, corpus, num_topics):
    """
    Extract theta matrix (document-topic distribution) from LDA model.
    
    Returns:
    - theta: 2D numpy array of shape (num_docs, num_topics)
             theta[i,j] = probability of topic j given document i
    """
    num_docs = len(corpus)
    theta = np.zeros((num_docs, num_topics))
    
    for doc_id, doc_topics in enumerate(lda_model.get_document_topics(corpus, minimum_probability=0)):
        for topic_id, prob in doc_topics:
            theta[doc_id, topic_id] = prob
    
    return theta

print("✓ Phi and theta extraction functions defined")

# Extract matrices for all languages
print("\n" + "="*70)
print("EXTRACTING PHI AND THETA MATRICES")
print("="*70)

phi_matrices = {}
theta_matrices = {}

for lang in main_languages:
    print(f"\n{lang}:")
    
    lda_model = lda_models[lang]
    num_topics = lda_model.num_topics
    vocab_size = len(dictionaries[lang])
    num_docs = len(corpora_data[lang])
    
    # Extract phi matrix (topics x vocabulary)
    print(f"  - Extracting phi matrix (topics x vocabulary)...")
    phi = extract_phi_matrix(lda_model, num_topics, dictionaries[lang], vocab_size)
    phi_matrices[lang] = phi
    print(f"    Shape: {phi.shape}")
    
    # Extract theta matrix (documents x topics)
    print(f"  - Extracting theta matrix (documents x topics)...")
    theta = extract_theta_matrix(lda_model, corpora_data[lang], num_topics)
    theta_matrices[lang] = theta
    print(f"    Shape: {theta.shape}")
    
print("\n" + "="*70)
print("✓ Matrices extracted successfully")
print("="*70)

✓ Phi and theta extraction functions defined

EXTRACTING PHI AND THETA MATRICES

en:
  - Extracting phi matrix (topics x vocabulary)...
    Shape: (3, 712)
  - Extracting theta matrix (documents x topics)...
    Shape: (1000, 3)

fr:
  - Extracting phi matrix (topics x vocabulary)...
    Shape: (5, 592)
  - Extracting theta matrix (documents x topics)...
    Shape: (1000, 5)

es:
  - Extracting phi matrix (topics x vocabulary)...
    Shape: (5, 750)
  - Extracting theta matrix (documents x topics)...
    Shape: (1000, 5)

de:
  - Extracting phi matrix (topics x vocabulary)...
    Shape: (3, 820)
  - Extracting theta matrix (documents x topics)...
    Shape: (1000, 3)

it:
  - Extracting phi matrix (topics x vocabulary)...
    Shape: (3, 973)
  - Extracting theta matrix (documents x topics)...
    Shape: (1000, 3)

pt:
  - Extracting phi matrix (topics x vocabulary)...
    Shape: (3, 851)
  - Extracting theta matrix (documents x topics)...
    Shape: (1000, 3)

ko:
  - Extracting phi ma

## 9. Save Results (Phi, Theta, and Comment IDs)

In [40]:
# Create output directory if it doesn't exist
output_dir = 'data'
os.makedirs(output_dir, exist_ok=True)

print("="*70)
print("SAVING RESULTS TO CSV")
print("="*70)

# For each language, save phi, theta, and top words per topic
for lang in main_languages:
    print(f"\n{lang}:")
    
    lang_df = language_data[lang]
    phi = phi_matrices[lang]
    theta = theta_matrices[lang]
    lda_model = lda_models[lang]
    dictionary = dictionaries[lang]
    
    num_topics = phi.shape[0]
    num_terms = phi.shape[1]
    
    # 1. THETA MATRIX with Comment IDs
    # Shape: (num_docs, num_topics), with document IDs
    print(f"  - Saving theta matrix (documents x topics)...")
    
    theta_df = pd.DataFrame(theta)
    theta_df.columns = [f'topic_{i}' for i in range(num_topics)]
    theta_df.insert(0, 'comment_id', lang_df['id'].values)
    theta_df.insert(1, 'langue', lang)
    
    theta_filename = os.path.join(output_dir, f'p02_lda_theta_{lang.lower().replace(" ", "_")}.csv')
    theta_df.to_csv(theta_filename, index=False, encoding='utf-8')
    print(f"    Saved: {theta_filename} (shape: {theta_df.shape})")
    
    # 2. PHI MATRIX
    # Shape: (num_topics, num_terms), with vocabulary terms
    print(f"  - Saving phi matrix (topics x vocabulary)...")
    
    # Create a dataframe with vocabulary terms and their probabilities across topics
    id2word = {v: k for k, v in dictionary.token2id.items()}
    
    # Transpose phi to get (vocab_size, num_topics)
    phi_transposed = phi.T  # Shape: (num_terms, num_topics)
    phi_df = pd.DataFrame(phi_transposed)
    phi_df.columns = [f'topic_{i}' for i in range(num_topics)]
    phi_df.insert(0, 'term_id', range(num_terms))
    phi_df.insert(1, 'term', [id2word.get(i, f'unknown_{i}') for i in range(num_terms)])
    phi_df.insert(2, 'langue', lang)
    
    phi_filename = os.path.join(output_dir, f'p02_lda_phi_{lang.lower().replace(" ", "_")}.csv')
    phi_df.to_csv(phi_filename, index=False, encoding='utf-8')
    print(f"    Saved: {phi_filename} (shape: {phi_df.shape})")
    
    # 3. TOP WORDS PER TOPIC
    print(f"  - Saving top words per topic...")
    
    top_words_data = []
    for topic_id in range(num_topics):
        terms = lda_model.show_topic(topic_id, topn=10)
        for rank, (term_name, prob) in enumerate(terms):
            term_id = dictionary.token2id.get(term_name, -1)  # Convert term to ID
            top_words_data.append({
                'langue': lang,
                'topic_id': topic_id,
                'rank': rank,
                'term_id': term_id,
                'term': term_name,
                'probability': prob
            })
    
    top_words_df = pd.DataFrame(top_words_data)
    top_words_filename = os.path.join(output_dir, f'p02_lda_top_words_{lang.lower().replace(" ", "_")}.csv')
    top_words_df.to_csv(top_words_filename, index=False, encoding='utf-8')
    print(f"    Saved: {top_words_filename} (shape: {top_words_df.shape})")
    
    # 4. DOCUMENT-TOPIC ASSIGNMENTS (argmax topic per document)
    print(f"  - Saving dominant topic per document...")
    
    dominant_topics = np.argmax(theta, axis=1)
    dominant_probs = np.max(theta, axis=1)
    
    doc_topics_df = pd.DataFrame({
        'comment_id': lang_df['id'].values,
        'langue': lang,
        'dominant_topic': dominant_topics,
        'dominant_topic_probability': dominant_probs
    })
    
    doc_topics_filename = os.path.join(output_dir, f'p02_lda_dominant_topics_{lang.lower().replace(" ", "_")}.csv')
    doc_topics_df.to_csv(doc_topics_filename, index=False, encoding='utf-8')
    print(f"    Saved: {doc_topics_filename} (shape: {doc_topics_df.shape})")

print("\n" + "="*70)
print("✓ All results saved successfully")
print("="*70)

SAVING RESULTS TO CSV

en:
  - Saving theta matrix (documents x topics)...
    Saved: data\p02_lda_theta_en.csv (shape: (1000, 5))
  - Saving phi matrix (topics x vocabulary)...
    Saved: data\p02_lda_phi_en.csv (shape: (712, 6))
  - Saving top words per topic...
    Saved: data\p02_lda_top_words_en.csv (shape: (30, 6))
  - Saving dominant topic per document...
    Saved: data\p02_lda_dominant_topics_en.csv (shape: (1000, 4))

fr:
  - Saving theta matrix (documents x topics)...
    Saved: data\p02_lda_theta_fr.csv (shape: (1000, 7))
  - Saving phi matrix (topics x vocabulary)...
    Saved: data\p02_lda_phi_fr.csv (shape: (592, 8))
  - Saving top words per topic...
    Saved: data\p02_lda_top_words_fr.csv (shape: (50, 6))
  - Saving dominant topic per document...
    Saved: data\p02_lda_dominant_topics_fr.csv (shape: (1000, 4))

es:
  - Saving theta matrix (documents x topics)...
    Saved: data\p02_lda_theta_es.csv (shape: (1000, 7))
  - Saving phi matrix (topics x vocabulary)...
    

## 10. Summary and Model Evaluation

In [41]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

summary_data = []

for lang in main_languages:
    print(f"\n{lang}:")
    print("-" * 70)
    
    lang_df = language_data[lang]
    lda_model = lda_models[lang]
    
    num_docs = len(lang_df)
    num_topics = lda_model.num_topics
    vocab_size = len(dictionaries[lang])
    perplexity = lda_model.log_perplexity(corpora_data[lang])
    coherence = final_coherence_scores[lang]
    
    print(f"  Documents (comments):  {num_docs:,}")
    print(f"  Number of topics:      {num_topics}")
    print(f"  Vocabulary size:       {vocab_size}")
    print(f"  Perplexity:            {perplexity:.4f}")
    print(f"  Coherence (U_Mass):    {coherence:.4f}")
    
    # Topic distribution
    print(f"\n  Topic distribution across corpus:")
    theta_mat = theta_matrices[lang]
    dominant_per_topic = np.bincount(np.argmax(theta_mat, axis=1), minlength=num_topics)
    for topic_id, count in enumerate(dominant_per_topic):
        pct = count / num_docs * 100
        bar = '█' * int(pct / 2)
        print(f"    Topic {topic_id:2d}: {bar:25s} {count:6,} docs ({pct:5.1f}%)")
    
    # Top 10 characteristic words per topic
    print(f"\n  Top 10 characteristic terms per topic:")
    for topic_id in range(num_topics):
        terms = lda_model.show_topic(topic_id, topn=10)
        term_list = ", ".join([f"{term} ({prob:.3f})" for term, prob in terms])
        print(f"    Topic {topic_id:2d}: {term_list}")
    
    summary_data.append({
        'Langue': lang,
        'Documents': num_docs,
        'Topics': num_topics,
        'Vocabulary': vocab_size,
        'Perplexity': perplexity,
        'Coherence': coherence,
    })

# Save summary
print("\n" + "="*70)
print("SAVING MODEL SUMMARY")
print("="*70)

summary_df = pd.DataFrame(summary_data)
summary_filename = os.path.join(output_dir, 'p02_lda_model_summary.csv')
summary_df.to_csv(summary_filename, index=False, encoding='utf-8')
print(f"✓ Summary saved to: {summary_filename}\n")

print(summary_df.to_string(index=False))

# Save optimization results
print("\n" + "="*70)
print("SAVING OPTIMIZATION RESULTS")
print("="*70)

optimization_data = []
for lang in main_languages:
    results = coherence_results[lang]
    for num_topics, coherence in zip(results['topic_range'], results['coherence_scores']):
        optimization_data.append({
            'langue': lang,
            'num_topics': num_topics,
            'coherence_u_mass': coherence,
            'is_optimal': (num_topics == results['optimal_topics'])
        })

optimization_df = pd.DataFrame(optimization_data)
optimization_filename = os.path.join(output_dir, 'p02_lda_topic_optimization.csv')
optimization_df.to_csv(optimization_filename, index=False, encoding='utf-8')
print(f"✓ Optimization results saved to: {optimization_filename}")

print("\n" + "="*70)
print("OUTPUT FILES GENERATED")
print("="*70)
print("\nFor each language, the following files have been created:")
print("  1. p02_lda_theta_[LANGUAGE].csv")
print("     - Document-topic distribution matrix")
print("     - Preserves comment_id for all rows")
print("  2. p02_lda_phi_[LANGUAGE].csv")
print("     - Topic-word distribution matrix")
print("  3. p02_lda_top_words_[LANGUAGE].csv")
print("     - Top 10 terms per topic with probabilities")
print("  4. p02_lda_dominant_topics_[LANGUAGE].csv")
print("     - Dominant topic assignment per document")
print("\nAdditional files:")
print("  5. p02_lda_model_summary.csv")
print("     - Model performance metrics per language")
print("  6. p02_lda_topic_optimization.csv")
print("     - Coherence scores for all tested topic numbers")

print("\n" + "="*70)
print("✓ ANALYSIS COMPLETE")
print("="*70)


FINAL SUMMARY

en:
----------------------------------------------------------------------
  Documents (comments):  1,000
  Number of topics:      3
  Vocabulary size:       712
  Perplexity:            -5.9358
  Coherence (U_Mass):    -1.4950

  Topic distribution across corpus:
    Topic  0: ███████████                  227 docs ( 22.7%)
    Topic  1: ██████████████████           374 docs ( 37.4%)
    Topic  2: ███████████████████          399 docs ( 39.9%)

  Top 10 characteristic terms per topic:
    Topic  0: apart (0.032), stay (0.028), locat (0.027), good (0.020), clean (0.015), pari (0.015), perfect (0.015), check (0.015), place (0.013), everyth (0.012)
    Topic  1: apart (0.025), stay (0.025), place (0.018), great (0.016), pari (0.015), locat (0.015), love (0.014), restaur (0.013), area (0.011), comfort (0.011)
    Topic  2: great (0.061), stay (0.038), place (0.033), locat (0.031), apart (0.026), host (0.022), recommend (0.020), pari (0.018), nice (0.015), would (0.015)

fr:

## Notes

### Sampling Strategy
- Each language is sampled with a fixed sample size (default: 1000 comments)
- This ensures computational efficiency while maintaining language-specific model training
- Modify `SAMPLE_SIZE_PER_LANGUAGE` to use all data or test with different sample sizes

### Topic Optimization
- Tested topic numbers from **MIN_TOPICS (3) to MAX_TOPICS (10)**
- Selected optimal number using **coherence score (U_Mass metric)**
- Lower/more negative coherence scores indicate better topic quality
- Results saved in `p02_lda_topic_optimization.csv` for analysis

### Matrices
- **Theta(documents × topics)**: Probability distribution of topics in each document
- **Phi (topics × vocabulary)**: Probability distribution of words in each topic
- Both matrices are normalized (rows sum to 1)

### Comment IDs Preservation
- All theta matrices include `comment_id` column for traceability
- Can be used to join with original reviews dataset for further analysis

### Next Steps
1. Visualize coherence trade-offs for different topic numbers
2. Correlate topic distributions with review ratings or other properties
3. Analyze temporal topic dynamics
4. Compare topic interpretation across languages

In [42]:
# ============================================================================
# PIPELINE RUNTIME & ENERGY METRICS
# ============================================================================

# Calculate total runtime
total_runtime_seconds = time.time() - PIPELINE_START_TIME
total_runtime_minutes = total_runtime_seconds / 60
total_runtime_hours = total_runtime_minutes / 60

# Calculate energy consumption
# Energy (Wh) = Power (W) × Time (hours)
energy_wh = POWER_WATTS * total_runtime_hours
energy_kwh = energy_wh / 1000

# Calculate carbon emissions
# CO2 (g) = Energy (kWh) × Grid intensity (kg CO2/kWh) × 1000
carbon_emissions_gco2e = energy_kwh * FR_GRID_KGCO2_PER_KWH * 1000

print("\n" + "="*70)
print("PIPELINE EXECUTION SUMMARY - RUNTIME & ENERGY METRICS")
print("="*70)

print(f"\n⏱️  RUNTIME:")
print(f"   Total duration: {total_runtime_hours:.2f}h ({total_runtime_minutes:.1f}m or {total_runtime_seconds:.0f}s)")

print(f"\n⚡ ENERGY CONSUMPTION:")
print(f"   Estimated power: {POWER_WATTS}W")
print(f"   Total energy: {energy_wh:.2f} Wh ({energy_kwh:.4f} kWh)")

print(f"\n🌍 CARBON FOOTPRINT:")
print(f"   Grid intensity: {FR_GRID_KGCO2_PER_KWH} kg CO2e/kWh (France)")
print(f"   Total emissions: {carbon_emissions_gco2e:.2f} gCO2e ({carbon_emissions_gco2e/1000:.4f} kg CO2e)")

print("\n" + "="*70)
print("✅ LDA PIPELINE COMPLETE")
print("="*70)

# Save metrics to CSV
metrics_df = pd.DataFrame([{
    'metric': 'Runtime (seconds)',
    'value': total_runtime_seconds
}, {
    'metric': 'Runtime (minutes)',
    'value': total_runtime_minutes
}, {
    'metric': 'Runtime (hours)',
    'value': total_runtime_hours
}, {
    'metric': 'Energy (Wh)',
    'value': energy_wh
}, {
    'metric': 'Energy (kWh)',
    'value': energy_kwh
}, {
    'metric': 'Carbon emissions (gCO2e)',
    'value': carbon_emissions_gco2e
}, {
    'metric': 'Carbon emissions (kg CO2e)',
    'value': carbon_emissions_gco2e / 1000
}])

metrics_filename = os.path.join(output_dir, 'p02_lda_pipeline_metrics.csv')
metrics_df.to_csv(metrics_filename, index=False, encoding='utf-8')
print(f"\n✓ Metrics saved to: {metrics_filename}")


PIPELINE EXECUTION SUMMARY - RUNTIME & ENERGY METRICS

⏱️  RUNTIME:
   Total duration: 0.67h (40.5m or 2428s)

⚡ ENERGY CONSUMPTION:
   Estimated power: 100W
   Total energy: 67.43 Wh (0.0674 kWh)

🌍 CARBON FOOTPRINT:
   Grid intensity: 0.06 kg CO2e/kWh (France)
   Total emissions: 4.05 gCO2e (0.0040 kg CO2e)

✅ LDA PIPELINE COMPLETE

✓ Metrics saved to: data\p02_lda_pipeline_metrics.csv
